### Dataset Preprocessing: Positive and Negative Dataset Preparation

This stage generated the positive and negative datasets required for the pulmonary nodule analysis framework.

The positive dataset was prepared for two separate tasks:
- **Segmentation:** Positive images containing annotated pulmonary nodules were used to train and evaluate the segmentation models.
- **Classification:** Positive images were combined with the negative dataset to train and evaluate the slice-level nodule-presence classifiers.

The negative dataset was generated from DICOM slices without nodule annotations in the LIDC-IDRI XML files. Patients included in either the positive segmentation or positive classification dataset were excluded from the negative dataset to prevent patient-level data leakage.

For both datasets, DICOM images were converted to Hounsfield Units (HU), clipped to the range of −1000 to 400 HU, normalized to 8-bit grayscale, and resized to 320 × 320 pixels. Corresponding binary masks were also generated.

The malignancy-score and polygon-area filters (malignancy score 3 and polygon area below 100 pixels) were applied later in the Main Ablation Study, after preprocessing. The 2.5D input construction using adjacent CT slices was also performed in the Main Ablation Study.

The positive segmentation dataset was used exclusively for segmentation, while the positive and negative datasets were used together for slice-level nodule-presence classification.

----------------

Positive dataset

In [ ]:

# ============================================================
# LIDC-IDRI DATA PREPROCESSING
#
# Processing steps:
#   1. Scan DICOM files and build an SOPInstanceUID dictionary.
#   2. Read radiologist annotations from XML files.
#   3. Convert raw DICOM pixel values to Hounsfield Units (HU).
#   4. Apply lung/nodule windowing [-1000, 400] HU.
#   5. Normalize images to the [0, 255] range.
#   6. Extract nodule ROIs and generate binary masks.
#   7. Resize full images and masks to 320 x 320.
#   8. Save images, masks, and metadata to CSV.
#
# Note:
#   Malignancy and polygon-area filtering are performed later
#   in the Main Ablation Study cells.
#
#   2.5D input construction is also performed later.
# ============================================================

import os
import csv
import cv2
import pydicom
import numpy as np
import xml.etree.ElementTree as ET

from tqdm import tqdm


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_ROOT = "PATH_TO_LIDC_IDRI"

TARGET_SIZE = 320

OUTPUT_IMG = f"resized_images{TARGET_SIZE}"
OUTPUT_MASK = f"resized_masks{TARGET_SIZE}"
CSV_PATH = f"resized_metadata{TARGET_SIZE}.csv"

# HU windowing parameters.
MIN_BOUND = -1000.0
MAX_BOUND = 400.0

os.makedirs(OUTPUT_IMG, exist_ok=True)
os.makedirs(OUTPUT_MASK, exist_ok=True)


# ============================================================
# 2. XML NAMESPACE REMOVAL
# ============================================================

def remove_xml_namespace(root):

    for element in root.iter():

        if "}" in element.tag:
            element.tag = element.tag.split("}", 1)[1]

    return root


# ============================================================
# 3. SCAN DICOM FILES
# ============================================================

print("Scanning DICOM files...")

dicom_paths = []

for root, dirs, files in os.walk(DATASET_ROOT):

    for filename in files:

        if filename.lower().endswith(".dcm"):

            dicom_paths.append(
                os.path.join(root, filename)
            )

print(f"Total DICOM: {len(dicom_paths)}")


# ============================================================
# 4. BUILD SOPINSTANCEUID DICTIONARY
# ============================================================

print("Building SOPInstanceUID dictionary...")

sop_dict = {}

dicom_read_errors = 0

for path in tqdm(dicom_paths, desc="Indexing DICOM files"):

    try:

        ds = pydicom.dcmread(
            path,
            stop_before_pixels=True
        )

        sop_uid = getattr(ds, "SOPInstanceUID", None)

        if sop_uid:
            sop_dict[sop_uid] = path

    except Exception:
        dicom_read_errors += 1

print(f"Indexed DICOM: {len(sop_dict)}")


# ============================================================
# 5. FIND XML ANNOTATION FILES
# ============================================================

print("Searching XML files...")

xml_files = []

for root, dirs, files in os.walk(DATASET_ROOT):

    for filename in files:

        if filename.lower().endswith(".xml"):

            xml_files.append(
                os.path.join(root, filename)
            )

print(f"Total XML: {len(xml_files)}")


# ============================================================
# 6. INITIALIZE PROCESSING COUNTERS
# ============================================================

image_id = 0

xml_parse_errors = 0
missing_patient_id = 0
missing_dicom_references = 0
pixel_read_errors = 0
invalid_polygons = 0
empty_resized_masks = 0
saved_images = 0


# ============================================================
# 7. PROCESS XML ANNOTATIONS
# ============================================================

print("Processing XML files...")

with open(
    CSV_PATH,
    "w",
    newline="",
    encoding="utf-8"
) as csv_file:

    csv_writer = csv.writer(csv_file)

    csv_writer.writerow([
        "image_name",
        "patient_id",
        "series_instance_uid",
        "sop_instance_uid",
        "slice_index",
        "malignancy",
        "polygon_area",
        "image_path",
        "mask_path"
    ])

    for xml_path in tqdm(xml_files, desc="Processing XML files"):

        # ----------------------------------------------------
        # Extract patient ID from the directory structure.
        # ----------------------------------------------------

        patient_id = None

        for part in xml_path.split(os.sep):

            if part.startswith("LIDC-IDRI-"):

                patient_id = part
                break

        if patient_id is None:

            missing_patient_id += 1
            continue

        # ----------------------------------------------------
        # Parse XML annotation.
        # ----------------------------------------------------

        try:

            tree = ET.parse(xml_path)
            root = remove_xml_namespace(tree.getroot())

        except Exception:

            xml_parse_errors += 1
            continue

        # ----------------------------------------------------
        # Process reading sessions.
        # ----------------------------------------------------

        for reading_session in root.findall(".//readingSession"):

            for nodule in reading_session.findall(
                "unblindedReadNodule"
            ):

                # Extract malignancy score.
                malignancy = 0

                characteristics = nodule.find("characteristics")

                if characteristics is not None:

                    malignancy_element = characteristics.find(
                        "malignancy"
                    )

                    if (
                        malignancy_element is not None
                        and malignancy_element.text
                    ):

                        try:
                            malignancy = int(
                                malignancy_element.text
                            )

                        except (ValueError, TypeError):
                            malignancy = 0

                # ------------------------------------------------
                # Process individual ROIs.
                # ------------------------------------------------

                for roi in nodule.findall("roi"):

                    sop_element = roi.find("imageSOP_UID")

                    if (
                        sop_element is None
                        or not sop_element.text
                    ):
                        continue

                    sop_uid = sop_element.text.strip()

                    if sop_uid not in sop_dict:

                        missing_dicom_references += 1
                        continue

                    dicom_path = sop_dict[sop_uid]

                    # --------------------------------------------
                    # Read DICOM and convert pixel values to HU.
                    # --------------------------------------------

                    try:

                        ds = pydicom.dcmread(dicom_path)

                        img = ds.pixel_array.astype(
                            np.float32
                        )

                        slope = float(
                            getattr(ds, "RescaleSlope", 1)
                        )

                        intercept = float(
                            getattr(ds, "RescaleIntercept", 0)
                        )

                        img = img * slope + intercept

                    except Exception:

                        pixel_read_errors += 1
                        continue

                    if img.ndim != 2:

                        pixel_read_errors += 1
                        continue

                    # --------------------------------------------
                    # Apply HU windowing.
                    # --------------------------------------------

                    img = np.clip(
                        img,
                        MIN_BOUND,
                        MAX_BOUND
                    )

                    # --------------------------------------------
                    # Normalize to [0, 255].
                    # --------------------------------------------

                    img = (
                        (img - MIN_BOUND)
                        / (MAX_BOUND - MIN_BOUND)
                    )

                    img = (img * 255).astype(np.uint8)

                    height, width = img.shape

                    # --------------------------------------------
                    # Extract polygon coordinates from XML.
                    # --------------------------------------------

                    points = []

                    for edge in roi.findall("edgeMap"):

                        try:

                            x_element = edge.find("xCoord")
                            y_element = edge.find("yCoord")

                            if (
                                x_element is None
                                or y_element is None
                                or x_element.text is None
                                or y_element.text is None
                            ):
                                continue

                            x = float(x_element.text)
                            y = float(y_element.text)

                            points.append((x, y))

                        except (ValueError, TypeError):
                            continue

                    if len(points) < 3:

                        invalid_polygons += 1
                        continue

                    polygon = np.array(
                        points,
                        dtype=np.int32
                    )

                    # --------------------------------------------
                    # Generate binary mask at original resolution.
                    # --------------------------------------------

                    mask = np.zeros(
                        (height, width),
                        dtype=np.uint8
                    )

                    cv2.fillPoly(
                        mask,
                        [polygon],
                        255
                    )

                    # --------------------------------------------
                    # Resize full image and mask.
                    # --------------------------------------------

                    resized_img = cv2.resize(
                        img,
                        (TARGET_SIZE, TARGET_SIZE),
                        interpolation=cv2.INTER_AREA
                    )

                    resized_mask = cv2.resize(
                        mask,
                        (TARGET_SIZE, TARGET_SIZE),
                        interpolation=cv2.INTER_NEAREST
                    )

                    # --------------------------------------------
                    # Skip masks that become empty after resizing.
                    # --------------------------------------------

                    if np.sum(resized_mask) == 0:

                        empty_resized_masks += 1
                        continue

                    # --------------------------------------------
                    # Extract DICOM metadata.
                    # --------------------------------------------

                    slice_index = int(
                        getattr(ds, "InstanceNumber", -1)
                    )

                    series_uid = getattr(
                        ds,
                        "SeriesInstanceUID",
                        ""
                    )

                    # --------------------------------------------
                    # Generate output filenames.
                    # --------------------------------------------

                    image_name = (
                        f"{patient_id}_slice{slice_index:04d}"
                        f"_img{image_id:05d}"
                    )

                    image_path = os.path.join(
                        OUTPUT_IMG,
                        image_name + ".png"
                    )

                    mask_path = os.path.join(
                        OUTPUT_MASK,
                        image_name + ".png"
                    )

                    # --------------------------------------------
                    # Save image and mask.
                    # --------------------------------------------

                    image_saved = cv2.imwrite(
                        image_path,
                        resized_img
                    )

                    mask_saved = cv2.imwrite(
                        mask_path,
                        resized_mask
                    )

                    if not image_saved or not mask_saved:
                        raise IOError(
                            f"Failed to save image or mask: {image_name}"
                        )

                    # Calculate polygon area at original resolution.
                    polygon_area = float(
                        np.sum(mask > 0)
                    )

                    # --------------------------------------------
                    # Save metadata.
                    # --------------------------------------------

                    csv_writer.writerow([
                        image_name,
                        patient_id,
                        series_uid,
                        sop_uid,
                        slice_index,
                        malignancy,
                        polygon_area,
                        image_path,
                        mask_path
                    ])

                    image_id += 1
                    saved_images += 1


# ============================================================
# 8. FINAL PROCESSING SUMMARY
# ============================================================

print("=" * 30)
print("Dataset Rebuild Completed")
print(f"Total whole images resized: {saved_images}")
print(f"CSV saved to: {CSV_PATH}")
print("=" * 30)


Scanning DICOM files...
Total DICOM: 258335
Building SOPInstanceUID dictionary...


100%|██████████| 258335/258335 [1:02:11<00:00, 69.23it/s] 


Indexed DICOM: 258335
Searching XML files...
Total XML: 1180
Processing XML files...


100%|██████████| 1180/1180 [15:09<00:00,  1.30it/s]


Dataset Rebuild Completed
Total whole images resized: 38229
CSV saved to: resized_metadata320.csv



-----------------

Negative dataset

In [ ]:
import os
import glob
import pydicom
import numpy as np
import cv2
import csv
import xml.etree.ElementTree as ET
from tqdm import tqdm
import warnings
import re
import pandas as pd

warnings.filterwarnings("ignore")

# ==========================================
# تنظیمات
# ==========================================
TARGET_SIZE = 320

LIDC_BASE_DIR = r"PATH_TO_LIDC_IDRI"

OUTPUT_IMG = f"resized_images_NegOnly_Full_{TARGET_SIZE}"
OUTPUT_MASK = f"resized_masks_NegOnly_Full_{TARGET_SIZE}"
CSV_PATH = f"resized_metadata_NegOnly_Full_{TARGET_SIZE}.csv"

os.makedirs(OUTPUT_IMG, exist_ok=True)
os.makedirs(OUTPUT_MASK, exist_ok=True)

MIN_BOUND = -1000.0
MAX_BOUND = 400.0

# ==========================================
# 0. Load POS patients (IMPORTANT)
# ==========================================
# Load metadata from the previously generated positive segmentation and positive classification datasets.
seg_df = pd.read_csv("segmentation_POS.csv")
cls_df = pd.read_csv("classification_POS.csv")

seg_patients = set(seg_df["patient_id"])
cls_patients = set(cls_df["patient_id"])

excluded_patients = seg_patients | cls_patients

print("\n===== EXCLUDED PATIENTS =====")
print("Segmentation:", len(seg_patients))
print("Classification:", len(cls_patients))
print("Total excluded:", len(excluded_patients))

# ==========================================
# helper
# ==========================================
def get_namespace(element):
    m = re.match('\{.*\}', element.tag)
    return m.group(0) if m else ''

# ==========================================
# 1. Mapping DICOMs
# ==========================================
print("\nPhase 1: Mapping DICOM files...")

sop_dict = {}
sop_to_patient = {}

dicom_files = glob.glob(os.path.join(LIDC_BASE_DIR, "**/*.dcm"), recursive=True)

for dcm_path in tqdm(dicom_files, desc="Indexing DICOMs"):
    try:
        ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        sop_uid = ds.SOPInstanceUID
        sop_dict[sop_uid] = dcm_path

        # extract patient id
        patient_id = None
        for part in dcm_path.split(os.sep):
            if part.startswith("LIDC-IDRI-"):
                patient_id = part
                break

        if patient_id is None:
            patient_id = "UNKNOWN"

        sop_to_patient[sop_uid] = patient_id

    except:
        continue

print(f"Total DICOMs indexed: {len(sop_dict)}")

# ==========================================
# 2. Identify POSITIVE slices from XML
# ==========================================
print("\nPhase 2: Parsing XMLs...")

xml_files = glob.glob(os.path.join(LIDC_BASE_DIR, "**/*.xml"), recursive=True)

positive_sops = set()

for xml_path in tqdm(xml_files, desc="Parsing XMLs"):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        ns = get_namespace(root)
    except:
        continue

    nodules = root.findall(f".//{ns}unblindedReadNodule") if ns else root.findall(".//unblindedReadNodule")

    for nodule in nodules:
        rois = nodule.findall(f".//{ns}roi") if ns else nodule.findall(".//roi")

        for roi in rois:
            sop_elem = roi.find(f"{ns}imageSOP_UID") if ns else roi.find("imageSOP_UID")
            if sop_elem is None:
                continue

            sop_uid = sop_elem.text
            if sop_uid in sop_dict:
                positive_sops.add(sop_uid)

print(f"Total Positive Slices: {len(positive_sops)}")

# ==========================================
# 3. Build NEGATIVE list (NO LEAKAGE)
# ==========================================
print("\nPhase 3: Building negative candidates...")

all_sops = set(sop_dict.keys())

# remove positive slices
candidate_negatives = all_sops - positive_sops

# remove POS patients
negative_sops = [
    sop for sop in candidate_negatives
    if sop_to_patient.get(sop, "UNKNOWN") not in excluded_patients
]

print(f"Final Negative SOPs: {len(negative_sops)}")

# ==========================================
# 4. Extraction
# ==========================================
print("\nPhase 4: Extracting negative images...")

csv_file = open(CSV_PATH, mode='w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)

csv_writer.writerow([
    'image_name', 'patient_id', 'series_uid', 'sop_uid',
    'slice_index', 'malignancy', 'polygon_area',
    'img_path', 'mask_path'
])

count = 0

for sop_uid in tqdm(negative_sops, desc="Generating Negatives"):

    dicom_path = sop_dict[sop_uid]

    try:
        ds = pydicom.dcmread(dicom_path)
        img = ds.pixel_array.astype(np.float32)

        intercept = getattr(ds, "RescaleIntercept", 0)
        slope = getattr(ds, "RescaleSlope", 1)
        img = img * slope + intercept

        # windowing
        img = np.clip(img, MIN_BOUND, MAX_BOUND)
        img = (img - MIN_BOUND) / (MAX_BOUND - MIN_BOUND)
        img = (img * 255).astype(np.uint8)

    except:
        continue

    if len(img.shape) > 2:
        img = img[:, :, 0]

    h, w = img.shape
    mask = np.zeros((h, w), np.uint8)

    resized_img = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(mask, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_NEAREST)

    patient_id = sop_to_patient.get(sop_uid, "UNKNOWN")
    slice_index = getattr(ds, "InstanceNumber", -1)
    series_uid = getattr(ds, "SeriesInstanceUID", "")

    name = f"{patient_id}_slice{slice_index:04d}_neg{count:05d}"

    img_out = os.path.join(OUTPUT_IMG, name + ".png")
    mask_out = os.path.join(OUTPUT_MASK, name + ".png")

    cv2.imwrite(img_out, resized_img)
    cv2.imwrite(mask_out, resized_mask)

    csv_writer.writerow([
        name, patient_id, series_uid, sop_uid,
        slice_index, 0, 0.0,
        img_out, mask_out
    ])

    count += 1

csv_file.close()

# ==========================================
# FINAL REPORT
# ==========================================
neg_patients = set([sop_to_patient[s] for s in negative_sops])

overlap = neg_patients & excluded_patients

print("\n==============================================")
print("NEGATIVE DATASET GENERATED")
print("Total samples:", count)
print("Total patients:", len(neg_patients))
print("Leakage check (must be 0):", len(overlap))
print("==============================================")


===== EXCLUDED PATIENTS =====
Segmentation: 524
Classification: 267
Total excluded: 791

Phase 1: Mapping DICOM files...


Indexing DICOMs: 100%|██████████| 258335/258335 [1:26:18<00:00, 49.88it/s]  


Total DICOMs indexed: 258335

Phase 2: Parsing XMLs...


Parsing XMLs: 100%|██████████| 1180/1180 [00:45<00:00, 25.97it/s]


Total Positive Slices: 16764

Phase 3: Building negative candidates...
Final Negative SOPs: 47196

Phase 4: Extracting negative images...


Generating Negatives: 100%|██████████| 47196/47196 [23:28<00:00, 33.51it/s]


NEGATIVE DATASET GENERATED
Total samples: 46498
Total patients: 219
Leakage check (must be 0): 0
